In [1]:
import pandas as pd

# Load ETH CSV
eth_df = pd.read_csv(r'feature_datasets\ETH_features.csv', parse_dates=True, index_col=0)

# Define X and y
eth_y = eth_df['Close'].shift(-1)          # next-day close price
eth_X = eth_df.drop(columns=['Close'])     # all other features

# Drop last row (y will be NaN)
eth_X = eth_X.iloc[:-1]
eth_y = eth_y.iloc[:-1]

# Optional: check shapes
print("ETH X shape:", eth_X.shape)
print("ETH y shape:", eth_y.shape)

ETH X shape: (915, 29)
ETH y shape: (915,)


## Basic 70-30 split

In [2]:
# 70% train / 30% test split (time-based)
split_idx = int(len(eth_X) * 0.7)

eth_X_train = eth_X.iloc[:split_idx].values
eth_X_test  = eth_X.iloc[split_idx:].values

eth_y_train = eth_y.iloc[:split_idx].values
eth_y_test  = eth_y.iloc[split_idx:].values

print("ETH X_train shape:", eth_X_train.shape)
print("ETH X_test shape:", eth_X_test.shape)
print("ETH y_train shape:", eth_y_train.shape)
print("ETH y_test shape:", eth_y_test.shape)

ETH X_train shape: (640, 29)
ETH X_test shape: (275, 29)
ETH y_train shape: (640,)
ETH y_test shape: (275,)


In [3]:
eth_test_dates = eth_df.index[-len(eth_X_test):]  

In [4]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Initialize scaler
scaler = MinMaxScaler()

# Fit on TRAIN only
eth_X_train_scaled = scaler.fit_transform(eth_X_train)

# Transform TEST using the same scaler
eth_X_test_scaled = scaler.transform(eth_X_test)

# Check shapes
print("Scaled X_train shape:", eth_X_train_scaled.shape)
print("Scaled X_test shape:", eth_X_test_scaled.shape)

Scaled X_train shape: (640, 29)
Scaled X_test shape: (275, 29)


In [5]:
# Initialize scaler for y
y_scaler = MinMaxScaler()

# Scale training target
eth_y_train_scaled = y_scaler.fit_transform(eth_y_train.reshape(-1,1)).flatten()

# Scale test target
eth_y_test_scaled = y_scaler.transform(eth_y_test.reshape(-1,1)).flatten()

In [6]:
def create_sequences(X, y, time_steps=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
        y_seq.append(y[i+time_steps])  # the next day after the sequence
    return np.array(X_seq), np.array(y_seq)

# Create sequences for ETH using scaled target
eth_X_train_seq, eth_y_train_seq = create_sequences(eth_X_train_scaled, eth_y_train_scaled, time_steps=30)
eth_X_test_seq, eth_y_test_seq = create_sequences(eth_X_test_scaled, eth_y_test_scaled, time_steps=30)

# Check shapes
print("ETH X_train_seq shape:", eth_X_train_seq.shape)
print("ETH y_train_seq shape:", eth_y_train_seq.shape)
print("ETH X_test_seq shape:", eth_X_test_seq.shape)
print("ETH y_test_seq shape:", eth_y_test_seq.shape)

ETH X_train_seq shape: (610, 30, 29)
ETH y_train_seq shape: (610,)
ETH X_test_seq shape: (245, 30, 29)
ETH y_test_seq shape: (245,)


### Train Bi_LSTM

In [7]:
from Bi_LSTM import BiLSTM, train_model, predict_model

In [8]:
# Get number of features from the sequences
input_size = eth_X_train_seq.shape[2]

# Initialize Bi-LSTM model
model = BiLSTM(input_size=input_size)

In [16]:
model = train_model(model, eth_X_train_seq, eth_y_train_seq, epochs=50,batch_size=32, lr=0.001
)

Epoch [10/50], Loss: 0.001190
Epoch [20/50], Loss: 0.001046
Epoch [30/50], Loss: 0.003319
Epoch [40/50], Loss: 0.002458
Epoch [50/50], Loss: 0.001233


In [17]:
eth_preds = predict_model(model, eth_X_test_seq)

# check
print("Predictions shape:", eth_preds.shape)
print("First 10 predictions:", eth_preds[:10])

Predictions shape: (245, 1)
First 10 predictions: [[0.25202248]
 [0.24962254]
 [0.2386636 ]
 [0.24190947]
 [0.2374613 ]
 [0.24928923]
 [0.23452948]
 [0.22335088]
 [0.21649745]
 [0.21119794]]


In [18]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Inverse scale predictions
eth_preds_actual = y_scaler.inverse_transform(eth_preds.reshape(-1,1))

# Skip first 30 because of 30-day sequence
eth_y_test_actual = eth_y_test[30:].reshape(-1,1)

# Calculate metrics
mse = mean_squared_error(eth_y_test_actual, eth_preds_actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(eth_y_test_actual, eth_preds_actual)
r2 = r2_score(eth_y_test_actual, eth_preds_actual)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")

MSE: 7538.23
RMSE: 86.82
MAE: 73.48
R2 Score: 0.8206
